# GCI In-Class Competition: Home Credit Default Risk
## Notebook 2: Optuna Hyperparameter Optimization (repaired)

**Status:** historical / optional. The shipped v4 members use the frozen params in
`pipeline/best_params_*_e2.json`; this notebook is the reproducible search that produced them.
It is not needed to rebuild or verify the submission.

### What was wrong in the original version (and is fixed here)
- It searched **GPU-only params** (`device='gpu'`, `tree_method='gpu_hist'`,
  `task_type='GPU'`), so it could not run on the CPU-only setup used by the final members.
- It tuned the **raw 101-column matrix** with no E2 enrichment, i.e. a different feature space
  than the one actually shipped.
- It used **3 folds** and **no target encoding**, while the final models use 5 folds + fold-local TE.
- `subsample` was suggested without `subsample_freq=1`, so LightGBM silently ignored bagging.
- `scale_pos_weight` was used during tuning but **not** in final training (notebook 3), so the
  search objective did not match the trained model.
- `trial.report(..., step=0)` only ran after all folds, so `MedianPruner` could never prune.
- CatBoost received float columns as `cat_features`, and XGBoost enabled categorical support on
  integer codes that were never `category` dtype.

### Protocol now
5-fold `StratifiedKFold(42)`, fold-local TE, CPU, `n_jobs`/`thread_count = 2`, the same
98-column E2 matrix as notebook 3, and per-fold pruning.

**Runtime:** 50 trials x 5 folds is still hours on CPU, so each study has a 2-hour timeout.
The timeout will usually bind before 50 trials; the best-so-far is saved either way.


In [ ]:
# 1. Setup
import numpy as np, pandas as pd, json, os, time, gc, warnings
import lightgbm as lgb, xgboost as xgb, catboost as cb
import optuna
from optuna.pruners import MedianPruner
from optuna.samplers import TPESampler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED = 42
N_SPLITS = 5
N_TRIALS = 50
TIMEOUT = 7200          # seconds per study
N_ESTIMATORS = 3000
EARLY_STOPPING = 200
N_JOBS = 2              # pinned for reproducibility (see AUDIT_FINDINGS.md)

print(f"Optuna {optuna.__version__} | LightGBM {lgb.__version__} | "
      f"XGBoost {xgb.__version__} | CatBoost {cb.__version__}")


In [ ]:
# 2. Mount Drive & locate the engineered data (same layout as notebooks 1/3/5)
from google.colab import drive
drive.mount('/content/drive')

CANDIDATES = ['/content/drive/MyDrive/GCI Competition',
              '/content/drive/MyDrive/gci_competition',
              '/content/drive/MyDrive/competition']
WORK_DIR = next((p for p in CANDIDATES if os.path.isdir(f'{p}/features')), None)
assert WORK_DIR is not None, f'no features/ folder under any of {CANDIDATES}'
os.makedirs(f'{WORK_DIR}/optuna_studies', exist_ok=True)
print('WORK_DIR:', WORK_DIR)

train = pd.read_parquet(f'{WORK_DIR}/features/train_engineered.parquet')
test = pd.read_parquet(f'{WORK_DIR}/features/test_engineered.parquet')
with open(f'{WORK_DIR}/features/feature_cols.json') as f:
    feature_cols = json.load(f)
with open(f'{WORK_DIR}/features/high_card_cols.json') as f:
    high_card_cols = json.load(f)
y = np.load(f'{WORK_DIR}/features/y_train.npy')

print(f"raw train {train.shape} | raw test {test.shape} | features {len(feature_cols)}")
print(f"positive rate {y.mean():.4f} | scale_pos_weight {(1-y.mean())/y.mean():.2f}")


In [ ]:
# 3. Same feature recipe as the shipped members: 101 base + 17 enrich - 18 dead = 98 columns
DEAD = [
    'FLAG_MOBIL','ORGANIZATION_TYPE','EXT_SOURCE_2_WAS_MISSING','CNT_FAM_MEMBERS_WAS_MISSING',
    'AMT_REQ_CREDIT_BUREAU_MON_WAS_MISSING','AMT_REQ_CREDIT_BUREAU_YEAR_WAS_MISSING',
    'EXT_SOURCE_1_WAS_MISSING','EXT_SOURCE_3_WAS_MISSING','FLAG_EMP_PHONE',
    'EXT_SOURCE_3_QBIN','OCCUPATION_TYPE_FREQ','NAME_HOUSING_TYPE_FREQ','NAME_FAMILY_STATUS_FREQ',
    'NAME_EDUCATION_TYPE_FREQ','NAME_INCOME_TYPE_FREQ','CODE_GENDER_FREQ','NAME_CONTRACT_TYPE_FREQ',
    'ORGANIZATION_TYPE_FREQ',
]

def enrich(df, base_cols):
    X = df.copy()
    eps = 1e-5
    ext = ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']
    filled = X[ext].replace(-999, np.nan)
    f = filled.fillna(filled.mean())
    age_years = (X['DAYS_BIRTH'].replace(-999, np.nan).abs() / 365.0).fillna(
        X['DAYS_BIRTH'].replace(-999, np.nan).abs().mean() / 365.0)
    for c in ext:
        X[f'{c}X_AGE'] = f[c] * age_years
    X['EXT_MEAN_X_AGE'] = f.mean(axis=1) * age_years
    X['EXT_2x3'] = f['EXT_SOURCE_2'] * f['EXT_SOURCE_3']
    X['EXT_1x2x3_NANSAFE'] = f['EXT_SOURCE_1'] * f['EXT_SOURCE_2'] * f['EXT_SOURCE_3']
    X['EXT_2_SQ'] = f['EXT_SOURCE_2'] ** 2
    X['EXT_3_SQ'] = f['EXT_SOURCE_3'] ** 2
    X['EXT_SUM'] = f.sum(axis=1)
    emp_years = X['DAYS_EMPLOYED'].replace(-999, np.nan).abs() / 365.0
    X['CREDIT_PER_INCOME_YEAR'] = X['AMT_CREDIT'] / ((X['AMT_INCOME_TOTAL'] + eps) * emp_years.fillna(emp_years.mean()) + eps)
    X['ANNUITY_PER_FAM'] = X['AMT_ANNUITY'] / (X['CNT_FAM_MEMBERS'].replace(0, 1) + eps)
    X['CREDIT_PER_FAM'] = X['AMT_CREDIT'] / (X['CNT_FAM_MEMBERS'].replace(0, 1) + eps)
    X['INCOME_PER_CHILD'] = X['AMT_INCOME_TOTAL'] / (X['CNT_CHILDREN'].clip(lower=0) + 1)
    X['NAN_COUNT'] = (df[base_cols] == -999).sum(axis=1).astype(np.int16)
    X['REGION_RATING_X_EXT_MEAN'] = X['REGION_RATING_CLIENT'] * f.mean(axis=1)
    return X

X0 = train[feature_cols].fillna(-999)
X_test0 = test[feature_cols].fillna(-999)
X = enrich(X0, feature_cols).drop(columns=[c for c in DEAD if c in X0.columns])
X_test = enrich(X_test0, feature_cols).drop(columns=[c for c in DEAD if c in X_test0.columns])
assert list(X.columns) == list(X_test.columns)
assert X.shape == (171202, 98), X.shape

# CatBoost requires integer cat codes; the parquet may hold them as float after fillna.
CATS = ['NAME_CONTRACT_TYPE','CODE_GENDER','NAME_INCOME_TYPE','NAME_EDUCATION_TYPE',
        'NAME_FAMILY_STATUS','NAME_HOUSING_TYPE','OCCUPATION_TYPE','FLAG_WORK_PHONE']
for c in CATS:
    if c in X.columns:
        X[c] = X[c].astype(np.int32)
        X_test[c] = X_test[c].astype(np.int32)

print(f"tuning matrix: {X.shape} | test {X_test.shape}")


In [ ]:
# 4. Folds + fold-local target encoding (identical to notebook 3 / shipped members)
# ORGANIZATION_TYPE is in DEAD, so only 6 of the 7 high-card columns are effective TE columns.
TE_COLS = [c for c in high_card_cols if c in X.columns]

def add_te(X_tr, X_va, X_te, y_tr, te_cols=TE_COLS, smoothing=10):
    X_tr, X_va, X_te = X_tr.copy(), X_va.copy(), X_te.copy()
    gs = pd.Series(y_tr, index=X_tr.index)
    gm = gs.mean()
    for col in te_cols:
        means = gs.groupby(X_tr[col]).mean()
        counts = X_tr[col].value_counts()
        sm = (means * counts + gm * smoothing) / (counts + smoothing)
        X_tr[f'{col}_TE'] = X_tr[col].map(sm).fillna(gm)
        X_va[f'{col}_TE'] = X_va[col].map(sm).fillna(gm)
        X_te[f'{col}_TE'] = X_te[col].map(sm).fillna(gm)
    return X_tr, X_va, X_te

FOLDS = list(StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED).split(X, y))

def cat_idx(Xc):
    return [list(Xc.columns).index(c) for c in CATS if c in Xc.columns]

print(f"folds: {len(FOLDS)} | TE cols ({len(TE_COLS)}): {TE_COLS}")


In [ ]:
# 5. LightGBM objective (CPU, 5 folds, fold-local TE, per-fold pruning)
def lgb_objective(trial):
    params = {
        'objective': 'binary',
        'metric': 'auc',
        'boosting_type': 'gbdt',
        'device': 'cpu',
        # Search ranges bracket the shipped E2 params but stay wide enough to move.
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.08, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 16, 96),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'min_child_samples': trial.suggest_int('min_child_samples', 20, 200),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'subsample_freq': 1,   # without this LightGBM ignores 'subsample' entirely
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
        'min_split_gain': trial.suggest_float('min_split_gain', 0.0, 1.0),
        'min_child_weight': trial.suggest_float('min_child_weight', 1e-3, 10.0, log=True),
        'random_state': SEED,
        'n_jobs': N_JOBS,
        'verbose': -1,
    }
    oof = np.zeros(len(X))
    fold_aucs = []
    for step, (tri, vai) in enumerate(FOLDS):
        X_tr, X_va, _ = add_te(X.iloc[tri], X.iloc[vai], X.iloc[vai], y[tri])
        model = lgb.LGBMClassifier(**params, n_estimators=N_ESTIMATORS)
        model.fit(X_tr, y[tri], eval_set=[(X_va, y[vai])], eval_metric='auc',
                  categorical_feature=cat_idx(X_tr),
                  callbacks=[lgb.early_stopping(EARLY_STOPPING, verbose=False)])
        oof[vai] = model.predict_proba(X_va)[:, 1]
        fold_aucs.append(roc_auc_score(y[vai], oof[vai]))
        trial.report(float(np.mean(fold_aucs)), step)
        if trial.should_prune():
            raise optuna.TrialPruned()
    return roc_auc_score(y, oof)

print("LightGBM objective defined")


In [ ]:
# 6. XGBoost objective (CPU hist; label-encoded cats are ordinal numeric, as in notebook 3)
def xgb_objective(trial):
    params = {
        'objective': 'binary:logistic',
        'eval_metric': 'auc',
        'tree_method': 'hist',
        'device': 'cpu',
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.08, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'min_child_weight': trial.suggest_float('min_child_weight', 1e-3, 30.0, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 1.0),
        'colsample_bylevel': trial.suggest_float('colsample_bylevel', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-4, 30.0, log=True),
        'gamma': trial.suggest_float('gamma', 0.0, 2.0),
        'n_jobs': N_JOBS,
        'verbosity': 0,
    }
    oof = np.zeros(len(X))
    fold_aucs = []
    for step, (tri, vai) in enumerate(FOLDS):
        X_tr, X_va, _ = add_te(X.iloc[tri], X.iloc[vai], X.iloc[vai], y[tri])
        dtr = xgb.DMatrix(X_tr, label=y[tri])
        dva = xgb.DMatrix(X_va, label=y[vai])
        model = xgb.train({**params, 'random_state': SEED}, dtr, num_boost_round=N_ESTIMATORS,
                          evals=[(dva, 'val')], early_stopping_rounds=EARLY_STOPPING,
                          verbose_eval=False)
        oof[vai] = model.predict(dva, iteration_range=(0, model.best_iteration + 1))
        fold_aucs.append(roc_auc_score(y[vai], oof[vai]))
        trial.report(float(np.mean(fold_aucs)), step)
        if trial.should_prune():
            raise optuna.TrialPruned()
    return roc_auc_score(y, oof)

print("XGBoost objective defined")


In [ ]:
# 7. CatBoost objective (CPU only; AUC early stopping is not reliable on GPU)
def cat_objective(trial):
    params = {
        'loss_function': 'Logloss',
        'eval_metric': 'AUC',
        'task_type': 'CPU',
        'thread_count': N_JOBS,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.10, log=True),
        'depth': trial.suggest_int('depth', 3, 6),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1.0, 10.0, log=True),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),
        'random_strength': trial.suggest_float('random_strength', 1e-3, 10.0, log=True),
        'border_count': trial.suggest_int('border_count', 32, 255),
        'random_seed': SEED,
        'verbose': False,
        'allow_writing_files': False,
    }
    oof = np.zeros(len(X))
    fold_aucs = []
    for step, (tri, vai) in enumerate(FOLDS):
        X_tr, X_va, _ = add_te(X.iloc[tri], X.iloc[vai], X.iloc[vai], y[tri])
        model = cb.CatBoostClassifier(**params, iterations=N_ESTIMATORS)
        model.fit(X_tr, y[tri], eval_set=(X_va, y[vai]), cat_features=cat_idx(X_tr),
                  early_stopping_rounds=EARLY_STOPPING, verbose=False)
        oof[vai] = model.predict_proba(X_va)[:, 1]
        fold_aucs.append(roc_auc_score(y[vai], oof[vai]))
        trial.report(float(np.mean(fold_aucs)), step)
        if trial.should_prune():
            raise optuna.TrialPruned()
    return roc_auc_score(y, oof)

print("CatBoost objective defined")


In [ ]:
# 8. Shared study runner: saves trials + best params (no fragile pickle of the Study object)
def run_study(name, objective, n_trials=N_TRIALS, timeout=TIMEOUT):
    study = optuna.create_study(
        direction='maximize',
        sampler=TPESampler(seed=SEED),
        pruner=MedianPruner(n_startup_trials=5, n_warmup_steps=1),
        study_name=name,
    )
    t0 = time.time()
    study.optimize(objective, n_trials=n_trials, timeout=timeout, show_progress_bar=True)
    elapsed = time.time() - t0
    states = [t.state.name for t in study.trials]
    print(f"\n{name}: {len(study.trials)} trials in {elapsed/60:.1f} min | "
          f"complete={states.count('COMPLETE')} pruned={states.count('PRUNED')}")
    print(f"best AUC {study.best_value:.5f} | {study.best_params}")

    study.trials_dataframe().to_csv(f'{WORK_DIR}/optuna_studies/{name}_trials.csv', index=False)
    with open(f'{WORK_DIR}/optuna_studies/best_params_{name}.json', 'w') as f:
        json.dump(study.best_params, f, indent=2)
    return study

print("study runner defined")


In [ ]:
# 9. Run the LightGBM study
study_lgb = run_study('lgb', lgb_objective)


In [ ]:
# 10. Run the XGBoost study
study_xgb = run_study('xgb', xgb_objective)


In [ ]:
# 11. Run the CatBoost study
study_cat = run_study('cat', cat_objective)


In [ ]:
# 12. Summary: combine the three best-param sets for notebook 3
best = {}
for key, study in [('lightgbm', study_lgb), ('xgboost', study_xgb), ('catboost', study_cat)]:
    best[key] = study.best_params
    print(f"\n{key}: OOF AUC {study.best_value:.5f}")
    for k, v in study.best_params.items():
        print(f"  {k}: {v}")

with open(f'{WORK_DIR}/optuna_studies/all_best_params.json', 'w') as f:
    json.dump(best, f, indent=2)

print("\nsaved all_best_params.json")
print("NOTE: notebook 3 also embeds frozen E2 params, so it does not depend on this file.")
